# PlantSR · Inpainting fine-tuning and evaluation (C2)
AAI 521 Group 1 · Ali · Google Colab · **GPU runtime (T4 or better)**

Week 1 selected Stable Diffusion 1.5 inpainting (`sd15_512_prompt`) and checked the LoRA training path with a 100-step trial. This notebook does the full fine-tuning and evaluation:

| Step | Section |
|---|---|
| Clone the repository, install, restore the dataset and the fixed Week 1 masks | 1–3 |
| Fine-tune LoRA adapters on the UNet attention layers (1,500 steps × 4 crops, checkpoint every 250 steps) | 4 |
| Pick the checkpoint with the best missing-region LPIPS on 24 validation crops (the Week 1 selection metric) | 5 |
| Install the selected adapter into the app and check it gives the same output there | 6 |
| Full validation and test evaluation: OpenCV Telea vs. pretrained SD 1.5 vs. fine-tuned, by mask type and size, paired comparison | 7–8 |
| Figures (visual comparison, boundary failures) and the result files for `reports/inpainting/` | 9–10 |

Training uses random 256 × 256 crops of the 663 full-size training images with random rectangle or free-form masks (5–20 % coverage), the same distribution as the fixed evaluation masks. Evaluation uses the same inference settings as the app (512 px, 25 DPM-Solver++ steps, guidance 5, seed 42, visible pixels composited back), so the numbers apply to the deployed model.

**Run time:** about 1 h 45 min on a T4 (training about 55 min, test evaluation about 25 min). **If Colab disconnects, choose Run all again:** training resumes from the last checkpoint and finished evaluations are read back from Drive. Everything is saved in `MyDrive/AAI521/Final_project/inpainting_c2/`.

## 1 · Clone the repository and install

In [ ]:
# Clone the branch and install the libraries (about 3 minutes).
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

env = os.environ.get
REPO_URL = "https://github.com/pwang4-LBNL/AAI521-Group1-PlantSR-Restoration.git"
BRANCH = env("PLANTSR_BRANCH", "feat/app")  # use "main" after the pull request is merged
CODE_DIR = Path(env("PLANTSR_CODE_DIR", "/content/AAI521-Group1-PlantSR-Restoration"))

if env("PLANTSR_SKIP_CLONE") != "1":
    if (CODE_DIR / ".git").is_dir():
        subprocess.run(["git", "-C", str(CODE_DIR), "fetch", "--quiet", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--quiet", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    else:
        subprocess.run(["git", "clone", "--quiet", "-b", BRANCH, REPO_URL, str(CODE_DIR)], check=True)
COMMIT = subprocess.run(["git", "-C", str(CODE_DIR), "log", "-1", "--format=%h %s"],
                        capture_output=True, text=True).stdout.strip() or "(not a git checkout)"
print("Code:", CODE_DIR, "·", COMMIT)
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

if env("PLANTSR_SKIP_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "app/requirements.txt"], check=True)

from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version

try:  # Colab's old torchao breaks peft (LoRA); the project does not use torchao
    if Version(version("torchao")) < Version("0.16.0"):
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "--quiet", "-y", "torchao"], check=True)
        importlib.invalidate_caches()
except PackageNotFoundError:
    pass

import PIL

if PIL.__version__ != version("Pillow"):
    print(f"Pillow {version('Pillow')} was installed but {PIL.__version__} is loaded. "
          "Restarting the runtime; then run the notebook again from the top.")
    os.kill(os.getpid(), 9)

for package in ("torch", "diffusers", "transformers", "peft", "accelerate", "lpips", "Pillow"):
    try:
        print(f"{package:14s} {version(package)}")
    except PackageNotFoundError:
        print(f"{package:14s} (not installed)")

## 2 · Mount Drive and set the run

In [ ]:
# Mount Drive and define the paths and settings for this run.
import csv
import json
import shutil
import time
import zipfile
from dataclasses import asdict

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

from src.train.inpaint_lora import LoraTrainConfig

PROJECT_DIR = Path(env("PLANTSR_PROJECT_DIR", "/content/drive/MyDrive/AAI521/Final_project"))
OUTPUT_DIR = PROJECT_DIR / "inpainting_c2"          # everything from this notebook (Drive)
CKPT_DIR = OUTPUT_DIR / "checkpoints"               # step_00250/, step_00500/, ... + train_state.pt
LORA_BEST = OUTPUT_DIR / "lora_best"                # the selected adapter; the app installs this folder
REPORT_DIR = OUTPUT_DIR / "reports" / "inpainting"  # copied to the repository as reports/inpainting/
LOCAL_DIR = Path(env("PLANTSR_LOCAL_DIR", "/content/plantsr"))
PREPARED_DIR = LOCAL_DIR / "prepared"
TEAM_ZIP = PROJECT_DIR / "PlantSR_prepared_for_team.zip"
TEAM_ZIP_DRIVE_ID = "1j_vx9Yuow1pu8MzYmbGY4udXBpB4dBx1"
WEEK1_DIR = PROJECT_DIR / "inpainting_week1"
MASKS_ZIP = WEEK1_DIR / "PlantSR_inpainting_masks_v1.zip"
SELECTION_CSV = WEEK1_DIR / "model_selection_val.csv"  # Week 1 scores for Telea and pretrained SD (reused)

APP = json.loads((CODE_DIR / "configs" / "app.json").read_text(encoding="utf-8"))["inpainting"]
INFERENCE = {k: APP[k] for k in ("work_res", "prompt", "negative_prompt", "steps", "guidance")}
INFERENCE["model_id"] = env("PLANTSR_MODEL_ID", APP["model_id"])
if env("PLANTSR_WORK_RES"):  # quick dry runs only
    INFERENCE.update(work_res=int(env("PLANTSR_WORK_RES")), steps=int(env("PLANTSR_SD_STEPS", "3")))

TRAIN = LoraTrainConfig(
    model_id=INFERENCE["model_id"], prompt=INFERENCE["prompt"], work_res=INFERENCE["work_res"],
    steps=int(env("PLANTSR_C2_STEPS", "1500")), batch_size=int(env("PLANTSR_C2_BATCH", "4")),
    checkpoint_every=int(env("PLANTSR_C2_EVERY", "250")), warmup_steps=int(env("PLANTSR_C2_WARMUP", "100")),
    rank=16, alpha=16, learning_rate=1e-4, crop_size=256, seed=42)
SEED = 42
SELECT_N = int(env("PLANTSR_C2_SELECT_N", "24"))  # validation crops used to pick the checkpoint
EVAL_LIMIT = int(env("PLANTSR_C2_EVAL_LIMIT", "0"))  # 0 = every validation/test crop
USE_LPIPS = env("PLANTSR_C2_LPIPS", "1") == "1"
SELECTED_STEP = None  # e.g. 1000 to overrule the automatic checkpoint choice after looking at section 5

for folder in (OUTPUT_DIR, CKPT_DIR, REPORT_DIR, LOCAL_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print(json.dumps({"train": asdict(TRAIN), "inference": INFERENCE}, indent=1, default=str)[:1500])
print("Outputs:", OUTPUT_DIR)

In [ ]:
# Check the GPU.
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    GPU = f"{props.name} · {props.total_memory / 1024**3:.1f} GB · compute capability {props.major}.{props.minor}"
else:
    GPU = "none (CPU)"
    print("No GPU detected. Choose Runtime → Change runtime type → T4 GPU, then rerun from the top.")
print("GPU", GPU)

## 3 · Restore the shared dataset and the fixed inpainting masks

In [ ]:
# Extract the team dataset and the Week 1 masks to local disk.
from IPython.display import Markdown, display

if not (PREPARED_DIR / "summary.json").is_file():
    zip_path = TEAM_ZIP
    if not zip_path.is_file():
        import gdown
        zip_path = LOCAL_DIR / TEAM_ZIP.name
        gdown.download(id=TEAM_ZIP_DRIVE_ID, output=str(zip_path), quiet=False)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(LOCAL_DIR)
if MASKS_ZIP.is_file() and not (PREPARED_DIR / "derived" / "inpainting_v1").is_dir():
    with zipfile.ZipFile(MASKS_ZIP) as archive:
        archive.extractall(LOCAL_DIR)


def read_csv(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


def inpaint_manifest(split):
    path = PREPARED_DIR / "derived" / "inpainting_v1" / f"inpainting_{split}.csv"
    return path if path.is_file() else None


TRAIN_ROWS = read_csv(PREPARED_DIR / "manifests" / "train.csv")
counts = {split: len(read_csv(PREPARED_DIR / "manifests" / f"{split}.csv")) for split in ("train", "val", "test")}
print("Images per split:", counts)
print("Fixed masks:", "Week 1 files" if inpaint_manifest("val") and inpaint_manifest("test") else
      "regenerated with the same seeded generator (identical to Week 1)")

## 4 · Fine-tune

LoRA adapters (rank 16) on the UNet attention projections; the VAE, text encoder and base UNet stay frozen. Each step encodes a batch of 4 crops and their masked copies, adds noise at a random timestep and trains the UNet to predict that noise (the standard inpainting objective). The learning rate warms up over 100 steps to 1e-4, gradients are clipped at 1.0, and mixed precision is used on the GPU.

A checkpoint (adapter + optimizer state) is written to Drive every 250 steps. **If the runtime disconnects, Run all again** and training continues after the last checkpoint.

In [ ]:
# Look at four training samples (crop, mask, what the model sees).
import matplotlib.pyplot as plt
import numpy as np

from src.data.inpaint_masks import overlay
from src.train.inpaint_lora import TrainCropDataset

preview = TrainCropDataset(PREPARED_DIR, TRAIN_ROWS, TRAIN.crop_size, TRAIN.work_res, seed=7)
fig, axes = plt.subplots(2, 4, figsize=(13, 6.6))
for i in range(4):
    sample = preview[i]
    image = ((sample["pixel_values"].permute(1, 2, 0).numpy() + 1) * 127.5).clip(0, 255).astype(np.uint8)
    mask = sample["mask"][0].numpy() > 0.5
    axes[0, i].imshow(overlay(image, mask))
    axes[0, i].set_title(f"{sample['mask_type']} · {sample['coverage']:.0%}", fontsize=9)
    seen = image.copy(); seen[mask] = 128
    axes[1, i].imshow(seen)
    axes[1, i].set_title("model input (masked)", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()
print(f"{len(preview)} training images · tensors {tuple(sample['pixel_values'].shape)} image, "
      f"{tuple(sample['mask'].shape)} mask")

In [ ]:
# Train (resumes automatically from the last checkpoint in Drive).
import gc

from src.inference.sd_inpaint import load_inpaint_pipeline
from src.train.inpaint_lora import list_checkpoints, train_lora

done = list_checkpoints(CKPT_DIR)
if done and done[-1][0] >= TRAIN.steps:
    print(f"Training already finished: {len(done)} checkpoints up to step {done[-1][0]}.")
    TRAIN_INFO = json.loads((OUTPUT_DIR / "train_summary.json").read_text()) if (OUTPUT_DIR / "train_summary.json").is_file() else {}
else:
    start = time.time()
    pipe = load_inpaint_pipeline(TRAIN.model_id)
    dataset = TrainCropDataset(PREPARED_DIR, TRAIN_ROWS, TRAIN.crop_size, TRAIN.work_res, TRAIN.seed)
    TRAIN_INFO = train_lora(pipe, dataset, TRAIN, CKPT_DIR, inference=INFERENCE)
    TRAIN_INFO["minutes_this_session"] = round((time.time() - start) / 60, 1)
    TRAIN_INFO["gpu"] = GPU
    (OUTPUT_DIR / "train_summary.json").write_text(json.dumps(TRAIN_INFO, indent=2), encoding="utf-8")
    del pipe, dataset
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
print(TRAIN_INFO)

In [ ]:
# Plot the training loss.
import pandas as pd

train_log = pd.read_csv(CKPT_DIR / "train_log.csv").drop_duplicates("step", keep="last").sort_values("step")
window = max(1, len(train_log) // 30)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.4))
ax1.plot(train_log.step, train_log.loss, alpha=0.25, label="per step")
ax1.plot(train_log.step, train_log.loss.rolling(window, min_periods=1).mean(), label=f"mean of {window} steps")
ax1.set(xlabel="step", ylabel="noise-prediction MSE", title="Training loss")
ax1.legend()
ax2.plot(train_log.step, train_log.learning_rate)
ax2.set(xlabel="step", ylabel="learning rate", title="Learning rate (warm-up, then constant)")
plt.tight_layout()
(REPORT_DIR / "figures").mkdir(parents=True, exist_ok=True)
fig.savefig(REPORT_DIR / "figures" / "training_loss.png", dpi=120)
plt.show()
quarter = max(1, len(train_log) // 4)
print(f"Mean loss: first quarter {train_log.loss.head(quarter).mean():.4f} · "
      f"last quarter {train_log.loss.tail(quarter).mean():.4f} · median {train_log.seconds.median():.2f}s/step")

## 5 · Choose the checkpoint on validation

Every checkpoint and the pretrained model generate the same 24 validation crops (spread over mask type and size) with the fixed masks and seed. The checkpoint with the lowest **missing-region LPIPS** is selected, the metric Week 1 used to choose the inference setting (masked SSIM if LPIPS is unavailable). The test split is not used until section 8.

In [ ]:
# Score the pretrained model and every checkpoint on the selection crops (results cached in Drive).
from src.data.benchmark_examples import BenchmarkExamples
from src.evaluation.inpaint_eval import (PRIMARY_METRIC, add_mask_info, evaluate, paired, stratified_ids, summary,
                                         wide)
from src.evaluation.inpaint_metrics import RESULT_FIELDS, LPIPSMetric
from src.inference.baselines import OpenCVInpaint
from src.inference.sd_inpaint import StableDiffusionInpaint

from diffusers.utils import logging as diffusers_logging

diffusers_logging.set_verbosity_error()  # hides "No LoRA keys associated to CLIPTextModel" (expected: UNet-only LoRA)
try:
    LPIPS = LPIPSMetric(device=DEVICE) if USE_LPIPS else None
except Exception as exc:  # noqa: BLE001 - LPIPS weights need a download
    print("LPIPS unavailable:", exc)
    LPIPS = None
RANK_METRIC = PRIMARY_METRIC if LPIPS is not None else "masked_ssim"


def sd_handler(lora=None):
    return StableDiffusionInpaint("sd15_finetuned" if lora else "sd15_pretrained", lora_path=lora,
                                  **{k: INFERENCE[k] for k in ("model_id", "work_res", "prompt", "negative_prompt",
                                                               "steps", "guidance")})


def cached(path, method, make_handler, examples, ids):
    """Evaluate `method` on `ids` unless the rows are already in `path` (so a restart skips finished work)."""
    path = Path(path)
    old = pd.read_csv(path) if path.is_file() else pd.DataFrame(columns=RESULT_FIELDS)
    have = old[(old.method == method)]
    if set(ids) <= set(have.sample_id):
        return have[have.sample_id.isin(ids)].to_dict("records")
    handler = make_handler()
    start = time.time()
    rows, _ = evaluate(handler, examples, ids, method=method, lpips_fn=LPIPS, seed=SEED)
    handler.unload()
    print(f"  {method:22s} {len(ids)} crops · {time.time() - start:.0f}s")
    keep = old[old.method != method]
    pd.concat([keep, pd.DataFrame(rows)], ignore_index=True).to_csv(path, index=False)
    return rows


val = BenchmarkExamples(PREPARED_DIR, "val", inpaint_manifest("val"), limit=None)
select_ids = stratified_ids(val, SELECT_N)
selection_csv = OUTPUT_DIR / "checkpoint_selection_val.csv"
rows = cached(selection_csv, "sd15_pretrained", sd_handler, val, select_ids)
for step, folder in list_checkpoints(CKPT_DIR):
    rows += cached(selection_csv, f"step_{step:05d}", lambda f=folder: sd_handler(f), val, select_ids)
selection = add_mask_info(wide(rows), val)
selection_summary = summary(selection)
display(selection_summary.round(4))

In [ ]:
# Plot the selection metric per checkpoint and pick the best one.
steps_df = selection_summary.drop(index="sd15_pretrained").copy()
steps_df["step"] = [int(m.split("_")[1]) for m in steps_df.index]
steps_df = steps_df.sort_values("step")
better_low = RANK_METRIC != "masked_ssim"
best_method = (steps_df[RANK_METRIC].idxmin() if better_low else steps_df[RANK_METRIC].idxmax())
if SELECTED_STEP is not None:
    best_method = f"step_{SELECTED_STEP:05d}"
BEST_STEP = int(best_method.split("_")[1])

fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, metric in zip(axes, [RANK_METRIC, "masked_psnr", "boundary_l1"]):
    if metric not in steps_df:
        continue
    ax.plot(steps_df.step, steps_df[metric], marker="o", label="fine-tuned checkpoints")
    ax.axhline(selection_summary.loc["sd15_pretrained", metric], color="grey", ls="--", label="pretrained")
    ax.axvline(BEST_STEP, color="green", alpha=0.4)
    ax.set(xlabel="training step", title=metric)
    ax.ticklabel_format(axis="y", useOffset=False)
axes[0].legend(fontsize=8)
plt.tight_layout()
fig.savefig(REPORT_DIR / "figures" / "checkpoint_selection.png", dpi=120)
plt.show()

pre, best = selection_summary.loc["sd15_pretrained", RANK_METRIC], selection_summary.loc[best_method, RANK_METRIC]
improved = best < pre if better_low else best > pre
print(f"Selected checkpoint: step {BEST_STEP} · {RANK_METRIC} {best:.4f} vs pretrained {pre:.4f} "
      f"({'better' if improved else 'NOT better'} on the {len(select_ids)} selection crops)")
if not improved:
    display(Markdown("**The fine-tuned adapter does not beat the pretrained model on the selection crops.** "
                     "Keep reading sections 7–8 before deciding; if it stays worse, report that and keep "
                     "`default_method = sd15_pretrained` in `configs/app.json`."))

from src.train.inpaint_lora import export_lora

export_lora(CKPT_DIR / best_method, LORA_BEST, {
    "selection": {"step": BEST_STEP, "metric": RANK_METRIC, "value": float(best), "pretrained_value": float(pre),
                  "crops": len(select_ids), "split": "val", "commit": COMMIT, "gpu": GPU}})
print("Exported to", LORA_BEST)

## 6 · Install the adapter into the app and check it

`scripts/install_checkpoints.py --inpaint-lora` copies the selected adapter to `checkpoints/inpaint/lora/`, the app registers **SD 1.5 inpainting · fine-tuned on PlantSR** and makes it the default. The same crop through the app's `restore()` must give exactly the pixels the evaluation produced.

In [ ]:
# Install the selected LoRA and compare the app's output with the evaluation output.
done = subprocess.run([sys.executable, "scripts/install_checkpoints.py", "--inpaint-lora", str(LORA_BEST)],
                      capture_output=True, text=True)
print("\n".join(l for l in done.stdout.splitlines() if "inpainting" in l or "LoRA" in l))
if done.returncode != 0:
    print(done.stderr[-2000:])
    raise RuntimeError("Installing the LoRA failed.")

from src.inference.factory import build_registry, load_config

config = load_config(CODE_DIR / "configs" / "app.json")
config["inpainting"].update({k: INFERENCE[k] for k in ("model_id", "work_res", "steps")})
config.update(include_pretrained=False)
registry = build_registry(config)
task = next(t for t in registry.describe() if t["task"] == "inpainting")
print("Inpainting methods:", [m["method"] for m in task["methods"]], "· default:", task["default_method"])

check_ids = select_ids[:2]
handler = sd_handler(LORA_BEST)
_, direct = evaluate(handler, val, check_ids, lpips_fn=None, seed=SEED, keep_outputs=2)
handler.unload()
same = []
for sid in check_ids:
    image, mask = val.degraded(sid, "inpainting")
    result = registry.restore(image, "inpainting", mask, seed=SEED, method="sd15_finetuned")
    same.append(result.success and np.array_equal(np.asarray(result.image), direct[sid]))
APP_OK = all(same) and task["default_method"] == "sd15_finetuned"
print("PASS" if APP_OK else "FAIL", " app output identical to the evaluation output:", same,
      "· version", registry.handler("inpainting", "sd15_finetuned").version)
for h in registry.gpu_handlers():
    h.unload()

## 7 · Full validation

All validation crops: OpenCV Telea (classical baseline), pretrained SD 1.5 and the fine-tuned adapter. Telea and pretrained SD were already scored with exactly these settings in Week 1 (`model_selection_val.csv`; the integration notebook confirmed the app reproduces them), so those rows are reused instead of regenerated.

In [ ]:
# Validation: reuse Week 1 rows for Telea / pretrained SD, generate the fine-tuned results.
from src.data.inpaint_masks import coverage_bin

WEEK1_NAMES = {"opencv_telea": "opencv_telea", f"sd15_{INFERENCE['work_res']}_prompt": "sd15_pretrained"}
val_ids = val.ids()[:EVAL_LIMIT] if EVAL_LIMIT else val.ids()
val_csv = OUTPUT_DIR / "val_results.csv"
val_rows = []
week1 = pd.read_csv(SELECTION_CSV) if SELECTION_CSV.is_file() else None
for week1_name, method in WEEK1_NAMES.items():
    reused = None
    if week1 is not None and LPIPS is not None:
        part = week1[(week1.method == week1_name) & week1.sample_id.isin(val_ids)].copy()
        if set(val_ids) <= set(part.sample_id) and PRIMARY_METRIC in set(part.metric_name):
            part["method"] = method
            reused = part.to_dict("records")
            print(f"  {method:22s} reused {part.sample_id.nunique()} crops from Week 1")
    if reused is None:
        make = (lambda: OpenCVInpaint("telea")) if method == "opencv_telea" else sd_handler
        reused = cached(val_csv, method, make, val, val_ids)
    val_rows += reused
val_rows += cached(val_csv, "sd15_finetuned", lambda: sd_handler(LORA_BEST), val, val_ids)
pd.DataFrame(val_rows).to_csv(val_csv, index=False)
val_table = add_mask_info(wide(val_rows), val)
display(summary(val_table).round(4))
display(paired(val_table, "sd15_finetuned", "sd15_pretrained").round(4))

## 8 · Test split (final numbers for the report)

Same three methods on every test crop with the fixed test masks. Nothing above used the test split.

In [ ]:
# Test: all three methods on the test split (about 25 minutes; cached in Drive).
test = BenchmarkExamples(PREPARED_DIR, "test", inpaint_manifest("test"), limit=None)
test_ids = test.ids()[:EVAL_LIMIT] if EVAL_LIMIT else test.ids()
test_csv = OUTPUT_DIR / "test_results.csv"
test_rows = []
test_rows += cached(test_csv, "opencv_telea", lambda: OpenCVInpaint("telea"), test, test_ids)
test_rows += cached(test_csv, "sd15_pretrained", sd_handler, test, test_ids)
test_rows += cached(test_csv, "sd15_finetuned", lambda: sd_handler(LORA_BEST), test, test_ids)
test_table = add_mask_info(wide(test_rows), test)
test_summary = summary(test_table)
display(test_summary.round(4))
display(Markdown("**By mask type and coverage** (missing-region LPIPS, lower is better; masked PSNR)"))
display(summary(test_table, by=("method", "mask_type")).round(4))
display(summary(test_table, by=("method", "coverage_bin")).round(4))
test_paired = paired(test_table, "sd15_finetuned", "sd15_pretrained")
test_vs_telea = paired(test_table, "sd15_finetuned", "opencv_telea")
display(Markdown("**Fine-tuned vs. pretrained (paired over the same crops)**"))
display(test_paired.round(4))
display(Markdown("**Fine-tuned vs. OpenCV Telea**"))
display(test_vs_telea.round(4))

## 9 · Figures: visual comparison and boundary failures

PSNR rewards blurry averages, so the classical fill often scores well on it while looking smeared; LPIPS and the images show whether texture and structure continue into the hole. The boundary failures are the test crops where the fine-tuned output differs most from the reference along the edge of the mask (`boundary_l1`), which shows where seams or colour shifts remain.

In [ ]:
# Side-by-side test crops and the worst boundary cases for the fine-tuned model.
from src.data.inpaint_masks import overlay


def run_methods(examples, ids):
    """Outputs of the three methods on a few crops (regenerated; same seed, so identical to the scored ones)."""
    outs = {}
    for method, make in (("opencv_telea", lambda: OpenCVInpaint("telea")), ("sd15_pretrained", sd_handler),
                         ("sd15_finetuned", lambda: sd_handler(LORA_BEST))):
        handler = make()
        _, outs[method] = evaluate(handler, examples, ids, method=method, seed=SEED, keep_outputs=len(ids))
        handler.unload()
    return outs


def grid(examples, ids, outs, title, path):
    cols = ["masked input", "OpenCV Telea", "SD 1.5 pretrained", "SD 1.5 fine-tuned", "reference"]
    fig, axes = plt.subplots(len(ids), 5, figsize=(14, 2.9 * len(ids)), squeeze=False)
    for r, sid in enumerate(ids):
        image, mask = examples.degraded(sid, "inpainting")
        info = examples.mask_info[sid]
        panels = [overlay(np.asarray(image), mask), outs["opencv_telea"][sid], outs["sd15_pretrained"][sid],
                  outs["sd15_finetuned"][sid], examples.reference(sid)]
        for c, panel in enumerate(panels):
            axes[r, c].imshow(panel)
            axes[r, c].axis("off")
            if r == 0:
                axes[r, c].set_title(cols[c], fontsize=10)
        axes[r, 0].text(2, 14, f"{info['mask_type']} {info['mask_coverage']:.0%}", color="white", fontsize=8,
                        bbox=dict(facecolor="black", alpha=0.5, pad=1))
    fig.suptitle(title, y=1.0)
    plt.tight_layout()
    fig.savefig(path, dpi=100, bbox_inches="tight", pil_kwargs={"quality": 88})  # JPG keeps the repo small
    plt.show()


ordered = sorted(test_ids, key=lambda s: (test.mask_info[s]["mask_type"], test.mask_info[s]["mask_coverage"]))
show_ids = [ordered[int(i * len(ordered) / 6 + len(ordered) / 12)] for i in range(6)] if len(ordered) >= 6 else ordered
outs = run_methods(test, show_ids)
grid(test, show_ids, outs, "Test crops: rectangle and free-form masks, small to large",
     REPORT_DIR / "figures" / "test_comparison.jpg")

ft = test_table[test_table.method == "sd15_finetuned"].dropna(subset=["boundary_l1"])
worst = ft.sort_values("boundary_l1", ascending=False).sample_id.head(4).tolist()
display(ft.set_index("sample_id").loc[worst, ["mask_type", "coverage_bin", "boundary_l1", "masked_bbox_lpips"
                                              if "masked_bbox_lpips" in ft else "masked_ssim", "masked_psnr"]].round(4))
outs_worst = run_methods(test, worst)
grid(test, worst, outs_worst, "Largest boundary error for the fine-tuned model (test)",
     REPORT_DIR / "figures" / "boundary_failures.jpg")

## 10 · Save the result files for the repository

In [ ]:
# Write reports/inpainting/ (summaries, per-image results, figures) and a ZIP to commit.
def save_split(table, rows, split):
    folder = REPORT_DIR / split
    folder.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(folder / f"per_image_{split}.csv", index=False)
    summary(table).round(6).to_csv(folder / f"summary_{split}.csv")
    summary(table, by=("method", "mask_type")).round(6).to_csv(folder / f"by_mask_type_{split}.csv")
    summary(table, by=("method", "coverage_bin")).round(6).to_csv(folder / f"by_coverage_{split}.csv")
    paired(table, "sd15_finetuned", "sd15_pretrained").round(6).to_csv(folder / f"paired_finetuned_vs_pretrained_{split}.csv",
                                                                       index=False)
    paired(table, "sd15_finetuned", "opencv_telea").round(6).to_csv(folder / f"paired_finetuned_vs_telea_{split}.csv",
                                                                    index=False)


save_split(val_table, val_rows, "val")
save_split(test_table, test_rows, "test")
shutil.copy2(CKPT_DIR / "train_log.csv", REPORT_DIR / "train_log.csv")
selection_summary.round(6).to_csv(REPORT_DIR / "checkpoint_selection_val.csv")
run_info = {"commit": COMMIT, "gpu": GPU, "train": asdict(TRAIN), "inference": INFERENCE, "train_info": TRAIN_INFO,
            "selected_step": BEST_STEP, "rank_metric": RANK_METRIC, "selection_crops": len(select_ids),
            "val_crops": len(val_ids), "test_crops": len(test_ids), "app_check": APP_OK,
            "lora_sha256": __import__("hashlib").sha256((LORA_BEST / "pytorch_lora_weights.safetensors").read_bytes()).hexdigest()}
(REPORT_DIR / "run_info.json").write_text(json.dumps(run_info, indent=2, default=str), encoding="utf-8")

zip_path = OUTPUT_DIR / "reports_inpainting.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(REPORT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, Path("reports/inpainting") / path.relative_to(REPORT_DIR))
print("Commit this ZIP (unzip in the repository root):", zip_path)
for path in sorted(REPORT_DIR.rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(OUTPUT_DIR))
print("\nAdapter for the app:", LORA_BEST)

## Next steps

1. Download `inpainting_c2/reports_inpainting.zip`, unzip it in the repository root and commit `reports/inpainting/`.
2. Rerun `notebooks/PlantSR_App_Integration.ipynb`: it finds `inpainting_c2/lora_best/` automatically, installs it, and the app shows **SD 1.5 inpainting · fine-tuned on PlantSR** as the default (the "Not installed" note disappears).
3. Use the test tables (section 8) and the two figures in the report's inpainting section.